# Mirror Thermal Tracking During the Night

Plots the air-to-mirror temperature difference ΔT = T_air − T_mirror during
the observing night, where:

- **T_air** = dome air near the mirror (ESS index 113, M1M3 vicinity)
- **T_mirror** = mirror bulk temperature proxy (`temperatureItem10`, ESS index 115)

A large positive ΔT means the air is warmer than the mirror (mirror lag after cooling).
A large negative ΔT means the mirror is warmer than the surrounding air (residual heat load).
The goal is ΔT ≈ 0 throughout the night to minimise seeing-degrading mirror turbulence.

**Reference time t₀:** evening nautical twilight (sun altitude = −12°) at Cerro Pachón

## Data sources
| Source | Topic / index | Field | Quantity |
|---|---|---|---|
| EFD | `lsst.sal.ESS.temperature`, index 113 | all `temperature*` fields | M1M3 vicinity air temp |
| EFD | `lsst.sal.ESS.temperature`, index 115 | `temperatureItem10` | Mirror bulk temperature proxy |

## Imports

In [ ]:
import datetime
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.coordinates import get_sun, AltAz, EarthLocation

from lsst_efd_client import EfdClient

%matplotlib inline

## Configuration

In [ ]:
# ── Observatory location ──────────────────────────────────────────────────────
RUBIN_LAT = -30.2444  # deg
RUBIN_LON = -70.7494  # deg
RUBIN_ELV = 2663.0  # m
rubin_loc = EarthLocation(
    lat=RUBIN_LAT * u.deg, lon=RUBIN_LON * u.deg, height=RUBIN_ELV * u.m
)

# ── Temperature sensors ───────────────────────────────────────────────────────
TEMP_TOPIC = "lsst.sal.ESS.temperature"

# Dome air temperature near the mirror
AIR_INDEX = 113
AIR_LABEL = "M1M3 vicinity air (ESS:113)"

# Mirror bulk temperature proxy
MIRROR_INDEX = 115
MIRROR_FIELD = "temperatureItem10"  # mirror bulk proxy
MIRROR_LABEL = "Mirror (ESS:115 item10)"

# ── Plot window ───────────────────────────────────────────────────────────────
# Hours after t₀ to show (full observing night ~10 h)
PLOT_HOURS_BEFORE_T0 = 0.5  # short pre-t0 context
PLOT_HOURS_AFTER_T0 = 10.0

RESAMPLE_FREQ = "5min"

# ── Single-night example ──────────────────────────────────────────────────────
SINGLE_NIGHT_DATE = "2026-03-15"

# ── Multi-night range ─────────────────────────────────────────────────────────
MULTI_NIGHT_END = "2026-05-07"  # last night (inclusive)
MULTI_NIGHT_WEEKS = 8

print("Configuration loaded.")

# ── Parquet cache ────────────────────────────────────────────────────
CACHE_MATRIX_FILE = "../data/mirror_thermal_matrix.parquet"
CACHE_T0_FILE = "../data/mirror_thermal_t0.parquet"
FORCE_REFETCH = False  # True → ignore cache, re-fetch everything
INCREMENTAL_REFETCH = False  # reserved for future incremental updates

In [ ]:
import pathlib as _pl

_cm = _pl.Path(CACHE_MATRIX_FILE)
_ct = _pl.Path(CACHE_T0_FILE)
_from_cache = False
if not FORCE_REFETCH and _cm.exists() and _ct.exists():
    _mat_df = pd.read_parquet(_cm)
    _t0_df = pd.read_parquet(_ct)
    all_dT_mirror = {
        str(idx): row.values.astype(float) for idx, row in _mat_df.iterrows()
    }
    dT_mirror_at_t0 = _t0_df["dT_at_t0"].to_dict()
    _from_cache = True
    print(f"✓ Loaded {len(all_dT_mirror)} nights from cache: {_cm}")
elif FORCE_REFETCH:
    print("FORCE_REFETCH=True — bypassing cache")
else:
    print(f"No cache at {_cm} — will fetch fresh data")

## Connect to EFD

In [ ]:
client = EfdClient("usdf_efd")
print("Connected to EFD")

## Helper Functions

In [ ]:
def evening_nautical_twilight(night_date_str):
    """Return UTC Time of evening nautical twilight (sun alt = −12°)."""
    t_start_search = Time(f"{night_date_str}T19:00:00", scale="utc")
    times = t_start_search + np.linspace(0, 10, 10_000) * u.hour
    frame = AltAz(obstime=times, location=rubin_loc)
    sun_alt = get_sun(times).transform_to(frame).alt.deg
    mask_above = sun_alt > -12.0
    crossings = np.where(np.diff(mask_above.astype(int)) < 0)[0]
    if len(crossings) == 0:
        warnings.warn(f"No nautical twilight found for {night_date_str}")
        return None
    i = crossings[0]
    dt_sec = (times[i + 1] - times[i]).to(u.s).value
    frac = (-12.0 - sun_alt[i]) / (sun_alt[i + 1] - sun_alt[i])
    return times[i] + TimeDelta(frac * dt_sec * u.s)


def mean_temp_cols(df):
    """Mean of all temperature* columns in a DataFrame."""
    cols = [c for c in df.columns if "temperature" in c.lower()]
    if not cols:
        raise ValueError(f"No temperature columns found. Available: {list(df.columns)}")
    return df[cols].mean(axis=1)


async def fetch_mirror_tracking(t_start, t_end):
    """
    Fetch air (ESS:113) and mirror (ESS:115 temperatureItem10) temperatures
    for the given window.  Returns (air_series, mirror_series) resampled to
    RESAMPLE_FREQ, both with UTC-aware DatetimeIndex.
    """
    # Air temperature near mirror
    df_air = await client.select_time_series(
        TEMP_TOPIC, fields="*", start=t_start, end=t_end, index=AIR_INDEX
    )
    if df_air.empty:
        raise RuntimeError(f"No air temperature data for ESS index {AIR_INDEX}.")
    air = mean_temp_cols(df_air).resample(RESAMPLE_FREQ).mean().rename(AIR_LABEL)
    print(f"  ESS:{AIR_INDEX} ({AIR_LABEL}): {len(df_air)} raw rows")

    # Mirror temperature
    df_mir = await client.select_time_series(
        TEMP_TOPIC, fields=[MIRROR_FIELD], start=t_start, end=t_end, index=MIRROR_INDEX
    )
    if df_mir.empty:
        raise RuntimeError(
            f"No mirror temperature data for ESS index {MIRROR_INDEX} field {MIRROR_FIELD}."
        )
    mirror = df_mir[MIRROR_FIELD].resample(RESAMPLE_FREQ).mean().rename(MIRROR_LABEL)
    print(f"  ESS:{MIRROR_INDEX} ({MIRROR_LABEL}): {len(df_mir)} raw rows")

    return air, mirror


print("Helpers defined.")

---
## Single Night — Air-to-Mirror ΔT vs Hours After Nautical Twilight

In [ ]:
if not _from_cache:
    t0_single = evening_nautical_twilight(SINGLE_NIGHT_DATE)
    print(f"Evening nautical twilight (t₀): {t0_single.iso} UTC")

    t_start_single = t0_single - TimeDelta(PLOT_HOURS_BEFORE_T0 * u.hour)
    t_end_single = t0_single + TimeDelta(PLOT_HOURS_AFTER_T0 * u.hour)
    print(f"Query window: {t_start_single.iso}  →  {t_end_single.iso} UTC")

    print("\nFetching temperatures…")
    air_single, mirror_single = await fetch_mirror_tracking(
        t_start_single, t_end_single
    )

In [ ]:
if not _from_cache:
    # ── Compute ΔT ────────────────────────────────────────────────────────────────
    common_idx = air_single.index.intersection(mirror_single.index)
    dT_single = (
        mirror_single.reindex(common_idx) - air_single.reindex(common_idx)
    ).dropna()

    t0_py = t0_single.to_datetime(timezone=datetime.timezone.utc)
    hours = np.asarray(
        (dT_single.index.tz_convert("UTC") - t0_py) / pd.Timedelta(hours=1), dtype=float
    )
    air_hours = np.asarray(
        (air_single.index.tz_convert("UTC") - t0_py) / pd.Timedelta(hours=1),
        dtype=float,
    )
    mir_hours = np.asarray(
        (mirror_single.index.tz_convert("UTC") - t0_py) / pd.Timedelta(hours=1),
        dtype=float,
    )

    # ── Plot ──────────────────────────────────────────────────────────────────────
    fig, (ax_dT, ax_T) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    # Top panel: ΔT
    ax_dT.plot(hours, dT_single.values, lw=1.5, color="steelblue")
    ax_dT.axhline(0, color="gray", lw=0.8, ls="--")
    ax_dT.axvline(0, color="red", lw=1.0, ls=":", label="t₀ (nautical twilight)")
    ax_dT.fill_between(
        hours,
        -1,
        1,
        where=np.ones_like(hours, dtype=bool),
        color="green",
        alpha=0.07,
        label="|ΔT| < 1 °C band",
    )
    ax_dT.set_ylabel("ΔT  (mirror − air)  [°C]", fontsize=10)
    ax_dT.set_title(
        f"Mirror Thermal Tracking — Night of {SINGLE_NIGHT_DATE}", fontsize=12
    )
    ax_dT.legend(fontsize=8, loc="upper right")
    ax_dT.grid(True, alpha=0.3)

    # Bottom panel: absolute temperatures
    ax_T.plot(air_hours, air_single.values, lw=1.2, color="steelblue", label=AIR_LABEL)
    ax_T.plot(
        mir_hours, mirror_single.values, lw=1.5, color="firebrick", label=MIRROR_LABEL
    )
    ax_T.axvline(0, color="red", lw=1.0, ls=":")
    ax_T.set_ylabel("Temperature [°C]", fontsize=10)
    ax_T.set_xlabel("Hours relative to evening nautical twilight (t₀)", fontsize=10)
    ax_T.legend(fontsize=8, loc="upper right")
    ax_T.grid(True, alpha=0.3)

    for ax in (ax_dT, ax_T):
        ax.set_xlim(-PLOT_HOURS_BEFORE_T0, PLOT_HOURS_AFTER_T0)
        ax.xaxis.set_major_locator(plt.MultipleLocator(2))
        ax.xaxis.set_minor_locator(plt.MultipleLocator(0.5))

    plt.tight_layout()
    plt.savefig(
        f"Mirror_tracking_single_{SINGLE_NIGHT_DATE}.pdf", dpi=150, bbox_inches="tight"
    )
    plt.show()

    idx_t0 = np.argmin(np.abs(hours))
    print(f"ΔT at t₀ = {dT_single.iloc[idx_t0]:+.2f} °C")
    print(f"Mean |ΔT| during night = {np.abs(dT_single.values).mean():.2f} °C")

### Single-night interpretation

The two-panel plot shows:

- **Top panel (ΔT):** The air-to-mirror temperature difference throughout the observing night.  The green shaded band (−1 to +2 °C) marks the target operating range — within this band the thermal gradient between the mirror surface and the surrounding air is small enough that mirror-induced seeing turbulence is subdominant.
- **Bottom panel (absolute temperatures):** The raw air and mirror temperature traces, useful for diagnosing whether the signal is driven by rapidly changing air (e.g. dome opening shock) or a slowly responding mirror.

**Physical interpretation:**
A positive ΔT at t₀ means the mirror's thermal mass has not yet cooled to ambient — the air cools faster than the mirror after sunset.  This is the most common pattern and typically self-corrects over 2–4 hours as the mirror equilibrates.  A negative ΔT (mirror warmer than air) can indicate residual heat load from the previous afternoon (e.g. insufficient daytime mirror cooling) or radiative cooling of the air outpacing the mirror.


---
## Multiple Nights — Overlay of ΔT Profiles

Each night's air-to-mirror ΔT curve is interpolated onto a common hour grid and
overlaid.  The nightly median and percentile envelopes are overplotted.

In [ ]:
t_end_multi = Time(f"{MULTI_NIGHT_END}T00:00:00", scale="utc")
t_start_multi = t_end_multi - TimeDelta(MULTI_NIGHT_WEEKS * 7 * u.day)

night_dates = [
    (t_start_multi + TimeDelta(i * u.day)).strftime("%Y-%m-%d")
    for i in range(MULTI_NIGHT_WEEKS * 7 + 1)
]
print(f"Processing {len(night_dates)} nights: {night_dates[0]} → {night_dates[-1]}")

In [ ]:
import asyncio

H_GRID = np.linspace(-PLOT_HOURS_BEFORE_T0, PLOT_HOURS_AFTER_T0, 500)
_BATCH_SIZE = 8


if not _from_cache:
    all_dT_mirror = {}  # night -> interpolated ΔT on H_GRID
    dT_mirror_at_t0 = {}  # night -> ΔT at h=0

    async def _process_night(night):
        t0 = evening_nautical_twilight(night)
        if t0 is None:
            return night, None, None
        t_s = t0 - TimeDelta(PLOT_HOURS_BEFORE_T0 * u.hour)
        t_e = t0 + TimeDelta(PLOT_HOURS_AFTER_T0 * u.hour)
        try:
            air, mirror = await fetch_mirror_tracking(t_s, t_e)
        except Exception:
            return night, None, None
        common = air.index.intersection(mirror.index)
        if len(common) < 10:
            return night, None, None
        dT = (mirror.reindex(common) - air.reindex(common)).dropna()
        t0_py = t0.to_datetime(timezone=datetime.timezone.utc)
        hrs = np.asarray(
            (dT.index.tz_convert("UTC") - t0_py) / pd.Timedelta(hours=1), dtype=float
        )
        dT_v = np.asarray(dT.values, dtype=float)
        sort_idx = np.argsort(hrs)
        interped = np.interp(
            H_GRID, hrs[sort_idx], dT_v[sort_idx], left=np.nan, right=np.nan
        )
        val_at_t0 = float(
            np.interp(0.0, hrs[sort_idx], dT_v[sort_idx], left=np.nan, right=np.nan)
        )
        return night, interped, val_at_t0

    for batch_start in range(0, len(night_dates), _BATCH_SIZE):
        batch = night_dates[batch_start : batch_start + _BATCH_SIZE]
        results = await asyncio.gather(
            *[_process_night(n) for n in batch], return_exceptions=True
        )
        for res in results:
            if isinstance(res, BaseException):
                continue
            night, interped, val_at_t0 = res
            if interped is not None:
                all_dT_mirror[night] = interped
                dT_mirror_at_t0[night] = val_at_t0
                print(f"  [{night}]  ΔT(t₀) = {val_at_t0:+.2f} °C")

    print(f"\nNights with usable data: {len(all_dT_mirror)} / {len(night_dates)}")

In [ ]:
# ── Save / update parquet cache ────────────────────────────────────────
import pathlib as _pl

_cm = _pl.Path(CACHE_MATRIX_FILE)
_ct = _pl.Path(CACHE_T0_FILE)
if _from_cache:
    print("Loaded from cache — skipping save")
elif all_dT_mirror:
    _mat_df = pd.DataFrame.from_dict(all_dT_mirror, orient="index")
    _mat_df.index.name = "night"
    _mat_df.to_parquet(_cm)
    _t0_series = pd.Series(dT_mirror_at_t0, name="dT_at_t0")
    _t0_series.index.name = "night"
    _t0_series.to_frame().to_parquet(_ct)
    print(f"✓ Saved {len(all_dT_mirror)} nights → {_cm}, {_ct}")
else:
    print("No data to cache.")

In [ ]:
if not all_dT_mirror:
    print("No data available — cannot plot.")
else:
    mat = np.vstack(list(all_dT_mirror.values()))
    dates_sorted = list(all_dT_mirror.keys())
    dT0_arr = np.array([dT_mirror_at_t0[d] for d in dates_sorted])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(mat, axis=0)
        p25 = np.nanpercentile(mat, 25, axis=0)
        p75 = np.nanpercentile(mat, 75, axis=0)
        p10 = np.nanpercentile(mat, 10, axis=0)
        p90 = np.nanpercentile(mat, 90, axis=0)

    vmin, vmax = np.nanpercentile(dT0_arr, [5, 95])
    cmap = plt.cm.RdYlGn_r
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    fig, ax_ov = plt.subplots(figsize=(10, 6))

    for i, (night, row) in enumerate(zip(dates_sorted, mat)):
        ax_ov.plot(H_GRID, row, lw=0.5, alpha=0.35, color=cmap(norm(dT0_arr[i])))

    ax_ov.fill_between(
        H_GRID, p10, p90, alpha=0.15, color="steelblue", label="10th–90th pct"
    )
    ax_ov.fill_between(H_GRID, p25, p75, alpha=0.25, color="steelblue", label="IQR")
    ax_ov.plot(H_GRID, med, lw=2.5, color="steelblue", label="Median")
    ax_ov.axhline(0, color="gray", lw=0.8, ls="--")
    ax_ov.axvline(0, color="red", lw=1.0, ls=":", label="t₀")
    ax_ov.fill_between(
        H_GRID,
        -2,
        1,
        where=np.ones_like(H_GRID, dtype=bool),
        color="green",
        alpha=0.07,
        label="Target: −2 to +1 °C",
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax_ov, label="ΔT at t₀  [°C]")

    ax_ov.set_xlim(-PLOT_HOURS_BEFORE_T0, PLOT_HOURS_AFTER_T0)
    ax_ov.xaxis.set_major_locator(plt.MultipleLocator(2))
    ax_ov.xaxis.set_minor_locator(plt.MultipleLocator(0.5))
    ax_ov.set_xlabel("Hours relative to evening nautical twilight (t₀)", fontsize=10)
    ax_ov.set_ylabel("ΔT  (mirror − air)  [°C]", fontsize=10)
    ax_ov.set_title(
        f"Mirror Thermal Tracking — {len(all_dT_mirror)} nights\n"
        f"{dates_sorted[0]}  to  {dates_sorted[-1]}",
        fontsize=11,
    )
    ax_ov.legend(fontsize=8, loc="upper right")
    ax_ov.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        f"Mirror_tracking_multinight_{dates_sorted[0]}_{dates_sorted[-1]}.pdf",
        dpi=150,
        bbox_inches="tight",
    )
    fig.savefig(
        "../EAS_SPIE2026/figures/mirror/mirrorconditioning.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

### Multi-night overlay interpretation

Each coloured trace is one night's ΔT(t) profile, interpolated onto a common hour grid and colour-coded by ΔT at t₀ (red = air much warmer than mirror at dome opening; green = well-matched).  The bold line is the nightly median; the shaded bands are the IQR and 10th–90th percentile envelope.

**Key observations:**
- The median ΔT starts positive at t₀ (~+1 °C), reflecting the mirror's thermal inertia relative to the rapidly cooling post-sunset air.
- Over the first 2–4 hours the median converges toward 0 °C as the mirror equilibrates with the dome air.
- Some nights show ΔT rising again in the second half of the night — this can indicate that mirror cooling is undershooting (mirror becomes colder than air) or that dome ventilation is bringing in warmer exterior air after an atmospheric temperature inversion breaks.
- Nights with large positive ΔT at t₀ (red) tend to remain offset from zero throughout — these may benefit from earlier or more aggressive mirror pre-cooling during the day.


In [ ]:
if all_dT_mirror:
    valid_dT0 = np.array([dT_mirror_at_t0[d] for d in dates_sorted])
    valid_dT0 = valid_dT0[np.isfinite(valid_dT0)]

    fig, ax_hist = plt.subplots(figsize=(5, 6))
    ax_hist.hist(
        valid_dT0,
        bins=20,
        orientation="horizontal",
        color="steelblue",
        edgecolor="white",
        alpha=0.8,
    )
    ax_hist.axhspan(-2, 1, color="green", alpha=0.07, label="Target: −2 to +1 °C")
    ax_hist.axhline(0, color="gray", lw=0.8, ls="--")
    ax_hist.axhline(
        np.nanmedian(valid_dT0),
        color="steelblue",
        lw=1.5,
        label=f"Median = {np.nanmedian(valid_dT0):+.2f} °C",
    )
    ax_hist.set_xlabel("Nights", fontsize=10)
    ax_hist.set_ylabel("ΔT at t₀  [°C]", fontsize=10)
    ax_hist.set_title("ΔT at dome-open time", fontsize=10)
    ax_hist.legend(fontsize=8)
    ax_hist.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.show()
    print(
        f"Median ΔT at t₀ = {np.nanmedian(valid_dT0):+.2f} °C  "
        f"(in target band: {np.mean((valid_dT0 >= -2) & (valid_dT0 <= 1)):.0%} of nights)"
    )

---
## Out-of-Spec Fraction per Night

**Spec:** mirror temperature must stay within the range T_air − 2 °C ≤ T_mirror ≤ T_air + 1 °C,
which means ΔT = T_air − T_mirror ∈ [−1, +2] °C.

Out-of-spec samples: ΔT < −1 °C (mirror too warm relative to air) **or** ΔT > +2 °C (mirror too cold).  
Only samples after t₀ (start of observing night, H > 0) are counted.

The bar chart shows the per-night out-of-spec percentage; red bars exceed 30 % of the night.
A linear trend fit tests whether the situation is improving or worsening over time.

In [ ]:
from scipy import stats as scipy_stats

# ── Spec limits (ΔT = T_air − T_mirror) ──────────────────────────────────────
DT_LO, DT_HI = -2.0, 1.0  # spec: mirror within +1/−2 °C of air

# Only post-t₀ (nighttime) H_GRID samples
night_mask = H_GRID >= 0

# ── Per-night out-of-spec fraction ────────────────────────────────────────────
oos_frac = {}  # night -> fraction out-of-spec
mean_dT_n = {}  # night -> mean ΔT during night

for night, dT_arr in all_dT_mirror.items():
    nightly = dT_arr[night_mask]
    finite = np.isfinite(nightly)
    n_valid = finite.sum()
    if n_valid < 10:
        continue
    n_oos = ((nightly[finite] < DT_LO) | (nightly[finite] > DT_HI)).sum()
    oos_frac[night] = n_oos / n_valid
    mean_dT_n[night] = float(np.nanmean(nightly))

oos_dates = np.array(sorted(oos_frac.keys()))
oos_values = np.array([oos_frac[d] for d in oos_dates])
oos_dt = pd.to_datetime(oos_dates)

# ── Previous-month summary ────────────────────────────────────────────────────
t_end_ts = pd.Timestamp(MULTI_NIGHT_END)
prev_month_start = t_end_ts.replace(day=1) - pd.DateOffset(months=1)
prev_month_end = t_end_ts.replace(day=1) - pd.DateOffset(days=1)
prev_mask = (oos_dt >= prev_month_start) & (oos_dt <= prev_month_end)

print(f"Previous month: {prev_month_start.date()} → {prev_month_end.date()}")
if prev_mask.any():
    prev_vals = oos_values[prev_mask]
    print(f"  Nights with data         : {prev_mask.sum()}")
    print(f"  Mean out-of-spec per night: {prev_vals.mean():.1%}")
    print(
        f"  Worst night              : {oos_dates[prev_mask][np.argmax(prev_vals)]}"
        f"  ({np.max(prev_vals):.1%})"
    )
    print(
        f"  Best night               : {oos_dates[prev_mask][np.argmin(prev_vals)]}"
        f"  ({np.min(prev_vals):.1%})"
    )
else:
    print("  No data for previous month.")

# ── Linear trend fit ──────────────────────────────────────────────────────────
import matplotlib.dates as mdates

oos_ordinal = mdates.date2num(oos_dt.to_pydatetime())
slope, intercept, r_val, p_val, se = scipy_stats.linregress(oos_ordinal, oos_values)
trend_line = slope * oos_ordinal + intercept
days_per_year = 365.25
print(
    f"\nLinear trend: {slope * days_per_year * 100:+.2f} percentage-points / year"
    f"   r = {r_val:.3f}   p = {p_val:.3f}"
)
trend_sig = (
    "statistically significant (p < 0.05)"
    if p_val < 0.05
    else "not statistically significant"
)
print(f"  → Trend is {trend_sig}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

bar_colors = ["tomato" if v > 0.30 else "steelblue" for v in oos_values]
ax.bar(
    oos_dt.to_pydatetime(),
    oos_values * 100,
    width=0.9,
    color=bar_colors,
    alpha=0.80,
    label="Per-night out-of-spec %",
)

# 7-night rolling mean
roll = pd.Series(oos_values, index=oos_dt).rolling(7, min_periods=3, center=True).mean()
ax.plot(
    roll.index.to_pydatetime(),
    roll.values * 100,
    color="darkorange",
    lw=2.0,
    label="7-night rolling mean",
)

ax.plot(
    oos_dt.to_pydatetime(),
    trend_line * 100,
    "--",
    color="firebrick",
    lw=1.5,
    label=f"Linear trend  ({slope * days_per_year * 100:+.1f} pp/yr, p={p_val:.2f})",
)

ax.axhline(30, color="orange", lw=0.8, ls=":", label="30 % reference")
ax.axvspan(
    prev_month_start,
    prev_month_end + pd.Timedelta(days=1),
    color="gold",
    alpha=0.10,
    label=f"Previous month ({prev_month_start.strftime('%b %Y')})",
)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=1))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_ylabel("Out-of-spec time [%]", fontsize=10)
ax.set_xlabel("Night date (UTC)", fontsize=10)
ax.set_title(
    f"Fraction of night where ΔT ∉ [{DT_LO}, {DT_HI}] °C  " f"({len(oos_frac)} nights)",
    fontsize=11,
)
ax.legend(fontsize=8, ncol=3, loc="upper left")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(
    f"Mirror_tracking_oos_fraction_{oos_dates[0]}_{oos_dates[-1]}.pdf",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

---
## Donut Blur FWHM: In-Spec vs Out-of-Spec Mirror Tracking

**Hypothesis:** out-of-spec mirror temperatures (ΔT outside [−1, +2] °C) produce enhanced
mirror boundary-layer turbulence, which inflates the donut blur FWHM measured by the AOS pipeline.

**Data sources:**
| Source | Table / field | Quantity |
|---|---|---|
| ConsDB | `cdb_lsstcam.visit1_quicklook.donut_blur_fwhm` | Donut blur half-power diameter (arcsec) |
| ConsDB | `cdb_lsstcam.exposure.obs_start` | Exposure UTC timestamp |
| This notebook | `all_dT_mirror` (EFD ESS:113/115) | Mirror ΔT at each visit epoch |

Each science exposure is matched to the nearest 5-min EFD ΔT sample (tolerance ≤ 15 min).
The Mann-Whitney U test compares the donut-blur distributions for in-spec and out-of-spec samples.

In [ ]:
import os
import sqlalchemy

# ── ConsDB credentials (same pattern as ConsDB_EFD_to_PSF_Effects_Diagnosis.ipynb) ───────────
_PGPASS_FILE = os.path.expanduser("~/.lsst/postgres-credentials.txt")
_CONSDB_HOST = "usdf-summitdb-logical-replica-svc.sdf.slac.stanford.edu"
_CONSDB_DB = "exposurelog"
_CONSDB_USER = "usdf"
_SCHEMA = "cdb_lsstcam"


def _load_pgpass(path, host, database, user):
    with open(path) as _f:
        for _line in _f:
            _line = _line.strip()
            if not _line or _line.startswith("#"):
                continue
            parts = _line.split(":")
            if len(parts) < 5:
                continue
            h, db, u_ = parts[0], parts[2], parts[3]
            pwd = ":".join(parts[4:])
            if h == host and db == database and u_ == user:
                return pwd
    raise ValueError(f"No credentials for {user}@{host}/{database}")


_db_pass = _load_pgpass(_PGPASS_FILE, _CONSDB_HOST, _CONSDB_DB, _CONSDB_USER)
_engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{_CONSDB_USER}:{_db_pass}@{_CONSDB_HOST}/{_CONSDB_DB}",
    connect_args={"connect_timeout": 30},
)


def _cq(sql):
    with _engine.connect() as _conn:
        return pd.read_sql_query(sqlalchemy.text(sql), _conn)


# ── Date range from the multi-night config ─────────────────────────────────────
_day_start = int(night_dates[0].replace("-", ""))
_day_end = int(night_dates[-1].replace("-", ""))

df_donut = _cq(
    f"""
    SELECT
        e.day_obs,
        e.obs_start,
        e.obs_start_mjd,
        e.band,
        e.exp_time,
        e.airmass,
        e.dimm_seeing,
        q.donut_blur_fwhm,
        q.psf_sigma_median,
        q.seeing_zenith_500nm_median,
        q.aos_fwhm
    FROM {_SCHEMA}.exposure e
    JOIN {_SCHEMA}.visit1_quicklook q
         ON q.day_obs = e.day_obs AND q.seq_num = e.seq_num
    WHERE e.day_obs BETWEEN {_day_start} AND {_day_end}
      AND e.img_type = 'science'
      AND q.donut_blur_fwhm IS NOT NULL
      AND q.donut_blur_fwhm > 0
    ORDER BY e.obs_start_mjd
"""
)

df_donut["obs_start_utc"] = pd.to_datetime(df_donut["obs_start"], utc=True)
df_donut = df_donut.sort_values("obs_start_utc").set_index("obs_start_utc")

print(f"Science exposures with donut_blur_fwhm : {len(df_donut)}")
print(f"Date range : {df_donut.index.min()}  →  {df_donut.index.max()}")
print(f"Band distribution:")
print(df_donut["band"].value_counts().to_string())
print(f"\ndonut_blur_fwhm [arcsec]:")
print(df_donut["donut_blur_fwhm"].describe().to_string())

In [ ]:
# ── Build absolute-time ΔT Series from per-night profiles ─────────────────────
# Re-derive t₀ for each stored night (pure astropy, no EFD calls)
_dT_records = []
for _night in sorted(all_dT_mirror.keys()):
    _t0 = evening_nautical_twilight(_night)
    if _t0 is None:
        continue
    _t0_utc = _t0.to_datetime(timezone=datetime.timezone.utc)
    for _h, _v in zip(H_GRID, all_dT_mirror[_night]):
        if np.isfinite(_v):
            _dT_records.append(
                {
                    "time_utc": _t0_utc + pd.Timedelta(hours=float(_h)),
                    "dT": float(_v),
                }
            )

dT_abs = pd.DataFrame(_dT_records).set_index("time_utc").sort_index()
print(
    f"Absolute-time ΔT series: {len(dT_abs)} samples over {len(all_dT_mirror)} nights"
)

# ── Nearest-neighbour join: each exposure → closest ΔT sample ─────────────────
merged_psf = pd.merge_asof(
    df_donut.sort_index(),
    dT_abs,
    left_index=True,
    right_index=True,
    tolerance=pd.Timedelta("15min"),
    direction="nearest",
).dropna(subset=["dT", "donut_blur_fwhm"])

merged_psf["in_spec"] = (merged_psf["dT"] >= DT_LO) & (merged_psf["dT"] <= DT_HI)
n_in_spec = merged_psf["in_spec"].sum()
n_out_spec = (~merged_psf["in_spec"]).sum()
print(
    f"\nMatched exposures : {len(merged_psf)}  "
    f"(in-spec: {n_in_spec}, out-of-spec: {n_out_spec})"
)

# ── Statistical comparison ─────────────────────────────────────────────────────
fwhm_in = merged_psf.loc[merged_psf["in_spec"], "donut_blur_fwhm"].dropna()
fwhm_out = merged_psf.loc[~merged_psf["in_spec"], "donut_blur_fwhm"].dropna()

print(f"\nDonut blur FWHM [arcsec]:")
print(
    f"  In-spec   (N={len(fwhm_in):4d})  median={fwhm_in.median():.3f}  "
    f"mean={fwhm_in.mean():.3f}  σ={fwhm_in.std():.3f}"
)
print(
    f"  Out-spec  (N={len(fwhm_out):4d})  median={fwhm_out.median():.3f}  "
    f"mean={fwhm_out.mean():.3f}  σ={fwhm_out.std():.3f}"
)

if len(fwhm_in) > 5 and len(fwhm_out) > 5:
    _mw_stat, _mw_p = scipy_stats.mannwhitneyu(
        fwhm_in, fwhm_out, alternative="two-sided"
    )
    _delta_med = fwhm_out.median() - fwhm_in.median()
    print(
        f"\n  Δmedian (out − in)         = {_delta_med:+.3f} arcsec  "
        f"({_delta_med / fwhm_in.median() * 100:+.1f} %)"
    )
    print(
        f"  Mann-Whitney U  p = {_mw_p:.4f}  "
        f"({'significant ✓' if _mw_p < 0.05 else 'not significant'})"
    )
else:
    _mw_p = np.nan
    print("  Not enough samples for statistical test.")

# ── Per-night median donut FWHM vs out-of-spec fraction ───────────────────────
merged_psf["night"] = merged_psf.index.normalize().strftime("%Y-%m-%d")
night_fwhm = (
    merged_psf.groupby("night")["donut_blur_fwhm"].median().rename("median_donut_fwhm")
)
night_oos_s = pd.Series(oos_frac)
nightly_joined = pd.concat(
    [night_fwhm, night_oos_s.rename("oos_frac")], axis=1
).dropna()
print(f"\nNights with both FWHM and OOS data: {len(nightly_joined)}")

In [ ]:
# ── Figure: three panels ──────────────────────────────────────────────────────
import matplotlib.patches as mpatches  # already imported above, safe to re-import

fig = plt.figure(figsize=(14, 11))
gs = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.30)
ax_sc = fig.add_subplot(gs[0, :])  # top row: full-width scatter
ax_box = fig.add_subplot(gs[1, 0])  # bottom-left: box plot
ax_nd = fig.add_subplot(gs[1, 1])  # bottom-right: night-level OOS vs FWHM

# ── Top: scatter ΔT vs donut FWHM ─────────────────────────────────────────────
_c_arr = merged_psf["in_spec"].map({True: "steelblue", False: "tomato"}).values
ax_sc.scatter(
    merged_psf["dT"],
    merged_psf["donut_blur_fwhm"],
    c=_c_arr,
    s=10,
    alpha=0.45,
    linewidths=0,
    rasterized=True,
)
ax_sc.axvspan(DT_LO, DT_HI, color="green", alpha=0.06, label="Spec region")
ax_sc.axvline(DT_LO, color="green", lw=0.8, ls="--")
ax_sc.axvline(DT_HI, color="green", lw=0.8, ls="--")

# Trend line (LOWESS-style: use binned medians)
_dT_bins = np.arange(
    merged_psf["dT"].min() // 0.5 * 0.5, merged_psf["dT"].max() + 0.5, 0.5
)
_bin_med = [
    merged_psf.loc[
        (merged_psf["dT"] >= lo) & (merged_psf["dT"] < lo + 0.5), "donut_blur_fwhm"
    ].median()
    for lo in _dT_bins[:-1]
]
_bin_ctr = _dT_bins[:-1] + 0.25
_valid_b = np.isfinite(_bin_med)
ax_sc.plot(
    _bin_ctr[_valid_b],
    np.array(_bin_med)[_valid_b],
    color="black",
    lw=1.8,
    ls="-",
    label="0.5 °C binned median",
    zorder=5,
)

in_p = mpatches.Patch(color="steelblue", alpha=0.7, label=f"In-spec  (N={n_in_spec})")
out_p = mpatches.Patch(color="tomato", alpha=0.7, label=f"Out-spec (N={n_out_spec})")
ax_sc.legend(
    handles=[
        in_p,
        out_p,
        mpatches.Patch(color="green", alpha=0.3, label="Spec region"),
        plt.Line2D([0], [0], color="black", lw=1.8, label="Binned median"),
    ],
    fontsize=8,
    ncol=4,
    loc="upper right",
)
ax_sc.set_xlabel("ΔT  (mirror − air)  [°C]", fontsize=10)
ax_sc.set_ylabel("Donut blur FWHM [arcsec]", fontsize=10)
ax_sc.set_title("Mirror ΔT vs donut blur FWHM (per exposure)", fontsize=11)
ax_sc.grid(True, alpha=0.3)

# ── Bottom-left: notched box plot ──────────────────────────────────────────────
_data = [fwhm_in.values, fwhm_out.values]
bp = ax_box.boxplot(
    _data,
    patch_artist=True,
    notch=False,
    medianprops={"color": "white", "lw": 2.5},
    flierprops={"marker": ".", "ms": 3, "alpha": 0.3},
)
for patch, col in zip(bp["boxes"], ["steelblue", "tomato"]):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)
ax_box.set_xticks([1, 2])
ax_box.set_xticklabels(
    [
        f"In-spec\nN={len(fwhm_in)}\nΔT ∈ [{DT_LO}, {DT_HI}] °C",
        f"Out-of-spec\nN={len(fwhm_out)}\nΔT < {DT_LO} or > {DT_HI} °C",
    ],
    fontsize=9,
)
ax_box.set_ylabel("Donut blur FWHM [arcsec]", fontsize=10)
ax_box.set_title("Donut FWHM distribution", fontsize=11)
ax_box.grid(True, alpha=0.3, axis="y")
if not np.isnan(_mw_p):
    _sig_str = "★ p < 0.05" if _mw_p < 0.05 else f"p = {_mw_p:.3f}"
    ax_box.text(
        1.5,
        ax_box.get_ylim()[1] * 0.98,
        _sig_str,
        ha="center",
        va="top",
        fontsize=9,
        color="firebrick",
        fontweight="bold" if _mw_p < 0.05 else "normal",
    )

# ── Bottom-right: per-night OOS fraction vs median donut FWHM ──────────────────
if len(nightly_joined) >= 3:
    _x = nightly_joined["oos_frac"].values
    _y = nightly_joined["median_donut_fwhm"].values
    _sc2 = ax_nd.scatter(
        _x * 100, _y, s=30, alpha=0.7, color="steelblue", edgecolors="none", zorder=3
    )
    _slp, _int, _r2, _p2, _ = scipy_stats.linregress(_x, _y)
    _xl = np.array([_x.min(), _x.max()])
    ax_nd.plot(
        _xl * 100,
        _slp * _xl + _int,
        "--",
        color="firebrick",
        lw=1.5,
        label=f"r = {_r2:.2f}  p = {_p2:.3f}",
    )
    ax_nd.legend(fontsize=8)
ax_nd.set_xlabel("Out-of-spec fraction per night [%]", fontsize=10)
ax_nd.set_ylabel("Median donut blur FWHM [arcsec]", fontsize=10)
ax_nd.set_title("Per-night: OOS fraction vs median FWHM", fontsize=11)
ax_nd.grid(True, alpha=0.3)

fig.suptitle(
    f"Mirror thermal tracking vs donut blur FWHM  "
    f"({night_dates[0]} → {night_dates[-1]})",
    fontsize=12,
    y=1.01,
)
plt.savefig(
    f"Mirror_tracking_donut_fwhm_{night_dates[0]}_{night_dates[-1]}.pdf",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

if not np.isnan(_mw_p):
    print(
        f"\nKey result: out-of-spec visits have median donut FWHM "
        f"{_delta_med:+.3f} arcsec ({_delta_med / fwhm_in.median() * 100:+.1f} %) "
        f"{'larger' if _delta_med > 0 else 'smaller'} than in-spec visits  "
        f"(Mann-Whitney p = {_mw_p:.4f})"
    )

---
## FCU Air–Mirror Temperature Difference vs Mirror Z-Gradient

The FCU (fan coil unit) air temperature is measured by the 96 `absoluteTemperature*`
channels of `lsst.sal.MTM1M3TS.thermalData` (the air blown onto the mirror by the FCUs).
The mirror bulk glass temperature is the mean over all temperature channels from
ESS indices 114–117 (`lsst.sal.ESS.temperature`, the thermocouples embedded in the glass).

Their difference — FCU air minus glass — is a direct driver of the mirror’s through-glass
axial thermal gradient.  This scatter plot shows per-night medians of that FCU ΔT against
the measured z-gradient (`z_gradient`, ∂T/∂z in °C/m) from `ThermocoupleAnalysis`
(ts_m1m3_utils).


In [ ]:
# ── Fetch FCU air temp (thermalData absoluteTemperature*) and mirror glass
# ── temp (ESS:114-117) in day-sized chunks to avoid transfer limits ─────────
THERMAL_TOPIC = "lsst.sal.MTM1M3TS.thermalData"
ABS_TEMP_COLS = [f"absoluteTemperature{i}" for i in range(96)]
GLASS_ESS_INDICES = [114, 115, 116, 117]
GLASS_ESS_TOPIC = "lsst.sal.ESS.temperature"
_RESAMP = "5min"

_t_start_multi_utc = Time(f"{MULTI_NIGHT_END}T00:00:00", scale="utc") - TimeDelta(
    MULTI_NIGHT_WEEKS * 7 * u.day
)
_t_end_multi_utc = Time(f"{MULTI_NIGHT_END}T12:00:00", scale="utc")

_CHUNK_DAYS = 1
_chunk_step = TimeDelta(_CHUNK_DAYS * u.day)

# ── FCU air: fetch thermalData in 1-day chunks, resample immediately ─────────
_fcu_chunks = []
_t = _t_start_multi_utc
while _t < _t_end_multi_utc:
    _t1 = min(_t + _chunk_step, _t_end_multi_utc)
    try:
        _df_td = await client.select_time_series(
            THERMAL_TOPIC, fields=ABS_TEMP_COLS, start=_t, end=_t1
        )
        if not _df_td.empty:
            _fcu_chunks.append(
                _df_td[ABS_TEMP_COLS].mean(axis=1).resample(_RESAMP).mean()
            )
    except Exception as _e:
        print(f"  thermalData chunk {_t.iso[:10]}: ERROR {_e}")
    _t = _t1

if not _fcu_chunks:
    print("No thermalData found.")
    df_fcu_glass = None
else:
    _fcu_air_5m = pd.concat(_fcu_chunks).sort_index()
    _fcu_air_5m = _fcu_air_5m[~_fcu_air_5m.index.duplicated(keep="last")]
    print(
        f"FCU air (thermalData): {len(_fcu_air_5m)} 5-min bins over "
        f"{MULTI_NIGHT_WEEKS} weeks"
    )

    # ── Mirror glass: ESS:114-117 (lower rate, single fetch per index) ──────
    _glass_chunks = []
    for _idx in GLASS_ESS_INDICES:
        try:
            _df_ess = await client.select_time_series(
                GLASS_ESS_TOPIC,
                fields="*",
                start=_t_start_multi_utc,
                end=_t_end_multi_utc,
                index=_idx,
            )
            if _df_ess.empty:
                print(f"  ESS:{_idx}: no data")
                continue
            _t_cols = [c for c in _df_ess.columns if c.startswith("temperatureItem")]
            _glass_chunks.append(_df_ess[_t_cols].mean(axis=1).resample(_RESAMP).mean())
            print(f"  ESS:{_idx}: {len(_df_ess)} rows  {len(_t_cols)} channels")
        except Exception as _e:
            print(f"  ESS:{_idx}: ERROR {_e}")

    if not _glass_chunks:
        print("No glass temperature data found.")
        df_fcu_glass = None
    else:
        _glass_all = pd.concat(_glass_chunks).sort_index()
        _glass_5m = _glass_all.groupby(_glass_all.index).mean()
        print(f"Mirror glass (ESS 114-117): {len(_glass_5m)} 5-min bins")

        df_fcu_glass = pd.DataFrame(
            {
                "fcu_air_temp": _fcu_air_5m,
                "mirror_glass_temp": _glass_5m,
            }
        ).dropna()
        df_fcu_glass["fcu_dT"] = (
            df_fcu_glass["fcu_air_temp"] - df_fcu_glass["mirror_glass_temp"]
        )
        # Clip unphysical fill values (|ΔT| > 50 °C)
        df_fcu_glass["fcu_dT"] = df_fcu_glass["fcu_dT"].where(
            df_fcu_glass["fcu_dT"].abs() < 50
        )
        print(f"Merged 5-min rows: {len(df_fcu_glass)}")

# ── Load z-gradient from the shared cache (written by MTAOS_Z4_Focus_Trends) ─
import pathlib as _pl2

_GRAD_CACHE = "../data/z4_m1m3_gradients.parquet"
if _pl2.Path(_GRAD_CACHE).exists():
    df_grad_fcu = pd.read_parquet(_GRAD_CACHE)[["z_gradient"]].copy()
    if df_grad_fcu.index.tzinfo is None:
        df_grad_fcu.index = df_grad_fcu.index.tz_localize("UTC")
    print(
        f"Loaded z-gradient cache: {len(df_grad_fcu)} rows "
        f"({df_grad_fcu.index.min().date()} → {df_grad_fcu.index.max().date()})"
    )
else:
    print(
        f"Gradient cache not found at {_GRAD_CACHE} — run MTAOS_Z4_Focus_Trends first."
    )
    df_grad_fcu = None

In [ ]:
# ── Resample to 10-min, filter to nighttime, scatter plot ───────────────────
if df_fcu_glass is None or df_grad_fcu is None:
    print("Insufficient data — skipping FCU ΔT vs z-gradient scatter.")
else:
    # 10-min resampling
    _fcu_10m = df_fcu_glass["fcu_dT"].resample("10min").mean()
    _grad_10m = df_grad_fcu["z_gradient"].resample("10min").mean()

    _fcu_r = _fcu_10m.reset_index()
    _grad_r = _grad_10m.reset_index()
    _fcu_r.columns = ["ts", "fcu_dT"]
    _grad_r.columns = ["ts", "z_gradient"]
    _fcu_r["ts"] = pd.to_datetime(_fcu_r["ts"], utc=True)
    _grad_r["ts"] = pd.to_datetime(_grad_r["ts"], utc=True)

    _merged = (
        pd.merge_asof(
            _fcu_r.sort_values("ts"),
            _grad_r.sort_values("ts"),
            on="ts",
            tolerance=pd.Timedelta("10min"),
            direction="nearest",
        )
        .dropna()
        .set_index("ts")
    )

    # ── Nighttime filter: within [t0 - 0.5h, t0 + 10h] per night ──────────
    _night_mask = pd.Series(False, index=_merged.index)
    for _nd in night_dates:
        _t0 = evening_nautical_twilight(_nd)
        if _t0 is None:
            continue
        import datetime as _dt

        _ts = (_t0 - TimeDelta(PLOT_HOURS_BEFORE_T0 * u.hour)).to_datetime(
            timezone=_dt.timezone.utc
        )
        _te = (_t0 + TimeDelta(PLOT_HOURS_AFTER_T0 * u.hour)).to_datetime(
            timezone=_dt.timezone.utc
        )
        _night_mask |= (_merged.index >= pd.Timestamp(_ts)) & (
            _merged.index <= pd.Timestamp(_te)
        )

    _merged_night = _merged[_night_mask].copy()
    print(f"Nighttime 10-min samples: {len(_merged_night)}")
    print(_merged_night[["fcu_dT", "z_gradient"]].describe().round(3))

    if len(_merged_night) >= 10:
        from scipy import stats as _scipy_stats

        _x = np.asarray(_merged_night["fcu_dT"].values, dtype=float)
        _y = np.asarray(_merged_night["z_gradient"].values, dtype=float)
        _msk = np.isfinite(_x) & np.isfinite(_y)
        _x, _y = _x[_msk], _y[_msk]

        _dates = _merged_night.index[_msk]
        _c_norm = (_dates.asi8 - _dates.asi8.min()) / (
            _dates.asi8.max() - _dates.asi8.min() + 1
        )

        _slp, _int, _r, _p, _ = _scipy_stats.linregress(_x, _y)
        _r2 = _r**2

        fig, ax = plt.subplots(figsize=(7, 5))
        sc = ax.scatter(
            _x,
            _y,
            c=_c_norm,
            cmap="plasma",
            s=4,
            alpha=0.25,
            edgecolors="none",
            rasterized=True,
            zorder=3,
        )
        _xl = np.array([_x.min(), _x.max()])
        ax.plot(
            _xl,
            _slp * _xl + _int,
            "--",
            color="firebrick",
            lw=1.8,
            zorder=4,
            label=f"Linear fit  r²={_r2:.2f}  p={_p:.3f}",
        )
        # 0.25 °C binned median
        _bins = np.arange(
            np.floor(_x.min() * 4) / 4, np.ceil(_x.max() * 4) / 4 + 0.25, 0.25
        )
        _bmed = [np.nanmedian(_y[(_x >= lo) & (_x < lo + 0.25)]) for lo in _bins[:-1]]
        _bctr = _bins[:-1] + 0.125
        _bv = np.isfinite(_bmed)
        ax.plot(
            _bctr[_bv],
            np.array(_bmed)[_bv],
            "-",
            color="black",
            lw=1.5,
            zorder=5,
            label="0.25 °C binned median",
        )
        ax.axhline(0, color="gray", lw=0.7, ls=":", alpha=0.5)
        ax.axvline(0, color="gray", lw=0.7, ls=":", alpha=0.5)
        cb = fig.colorbar(sc, ax=ax, pad=0.02)
        cb.set_label("Time (early → late in study period)", fontsize=9)
        ax.set_xlabel("FCU air − mirror glass  ΔT  [°C]  (10-min bin)", fontsize=10)
        ax.set_ylabel("Mirror z-gradient  ∂T/∂z  [°C/m]  (10-min bin)", fontsize=10)
        ax.set_title(
            f"FCU ΔT vs mirror z-gradient  (nighttime, 10-min bins)\n"
            f"N={len(_x):,}  |  {night_dates[0]} → {night_dates[-1]}",
            fontsize=11,
        )
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            f"Mirror_FCU_dT_vs_z_gradient_nighttime_10min_{night_dates[0]}_{night_dates[-1]}.pdf",
            dpi=150,
            bbox_inches="tight",
        )
        plt.show()
        print(f"\nFit: slope={_slp:.4f} (°C/m)/(°C)  r²={_r2:.3f}  p={_p:.4g}")
    else:
        print("Not enough nighttime data.")

### Summary of mirror thermal tracking performance

Over the 8-week study period (Mar–May 2026):

| Metric | Value |
|---|---|
| Qualifying nights | ~55 |
| Median ΔT at t₀ (air − mirror) | **+1.23 °C** |
| Fraction of nights in target band (−1 to +2 °C) at t₀ | **83%** |

The median positive offset at dome opening is physically expected: the mirror's large thermal mass (about 7,000 kg for M1M3) means it cannot cool as quickly as the surrounding air after sunset.  The system is within the target band on the majority of nights, suggesting the daytime mirror pre-conditioning (via the dome cooling system) is performing well.

**Nights outside the target band** (17%) warrant investigation — they may correspond to days with elevated solar loading, anomalous dome temperatures, or AHU outages that prevented adequate pre-cooling.  Cross-referencing with the HVAC daytime conditioning notebook (`HVAC_DaytimeThermalConditioning.ipynb`) can confirm whether poor daytime conditioning correlates with poor mirror tracking at night.
